# Strands Swarm + A2A: a team of agents, each with its own identity

Two ways [Strands](https://strandsagents.com) agents on Amazon Bedrock work together, each with four
things from Highflame:

| Pattern | How agents cooperate | What identity adds |
| --- | --- | --- |
| **Swarm** ([Strands docs](https://strandsagents.com/docs/user-guide/concepts/multi-agent/swarm/)) | Specialists hand off to each other autonomously, sharing context, with no central controller | Every hand-off is a tool call, so it is checked and attributed like any other. Each specialist works under its own short-lived credential |
| **A2A** ([Agent-to-Agent protocol](https://strandsagents.com/docs/user-guide/concepts/multi-agent/agent-to-agent/)) | An agent in another process, or another company, is called over HTTP | The caller presents its Highflame credential on the wire. The remote agent verifies who is calling and what they are allowed before it does any work, and runs its own guardrails under its own identity |

1. **Identity.** A team lead you register in Studio registers every specialist, local or remote, as
   an identity of its own, so every decision Highflame records names the agent that caused it.
2. **Authorization.** Two layers. The team lead delegates each specialist a short-lived credential
   narrowed to the scope its job needs, and each specialist's allow-list decides which tools it may
   call. The remote agent also checks the caller's credential at its front door.
3. **Runtime guardrails.** Four Strands hooks on every agent check its prompts, tool calls, tool
   results and replies before they proceed. Content coming back from the remote agent is a tool
   result, so it is checked too.
4. **Telemetry.** Every decision is attributed to the agent that caused it, and each delegated
   credential records who issued it.

> A single Strands agent, and an orchestrator that calls specialists as tools, are in
> [`strands_bedrock_agent_identity.ipynb`](strands_bedrock_agent_identity.ipynb). One Studio identity
> can serve both notebooks.

## Setup

### 1. Register the team lead in Studio

This notebook runs a team whose lead you register by hand, in the UI. It is the root of trust: the
only identity created outside this notebook, and the one every specialist here is registered and
delegated by. It never runs an agent itself.

Already registered an agent for `strands_bedrock_agent_identity.ipynb` or
`langgraph_agent_identity.ipynb`? Use that identity: add `refunds:read` to its credential policy,
restart the kernel, and skip to step 2.

Its authority ceiling is a **credential policy**, attached at registration. Create the policy
first, then register the identity and pick it.

**Studio → Registry → Policies → Create Policy**

| Field | Value |
| --- | --- |
| Name | `support-team-cred`, or any name |
| Allowed scopes | `nhi:manage`, `tools:read`, `tools:execute`, `orders:read`, `kb:read`, `refunds:read` |
| Allowed grant types | must include `api_key` |

**Studio → Registry → Agents → Inventory → Register Identity**

| Field | Value |
| --- | --- |
| Name | `Support Team Lead` |
| Identity type | `agent` |
| Sub type | `orchestrator` |
| Trust level | `first_party` |
| Credential policy | the policy above |

What each scope is for:

| Scope | Why the team lead needs it |
| --- | --- |
| `nhi:manage` | Registers the four specialists. |
| `tools:read`, `tools:execute` | A credential needs `tools:read` to send a prompt and `tools:execute` to call a tool, hand-offs included. The team lead can only delegate what it holds. |
| `orders:read`, `kb:read`, `refunds:read` | Each specialist's domain scope. `refunds:read` is what the remote agent requires of its callers. |

**List the scopes; do not leave the field empty.** An empty *Allowed scopes* field puts no
restriction on the key, but it also gives the key no scope to hand on: every delegation is then
refused with `invalid_scope: requested scopes are not available for delegation`. A scope missing
from the list is worse, because nothing fails: it is silently dropped from the specialist's
credential, and without `refunds:read` the remote agent refuses triage with a `403`. The setup cell
checks the key's scopes before anything is registered.

**`nhi:manage` is the one people miss.** It is what lets a key register other identities. Leave it
out and the identity is still created, the key still works and `whoami()` still succeeds — then
the first `agents.register()` fails with `403 token missing nhi:manage scope`.

The key is shown **once**, at creation.

No allow-list is needed on the team lead, because it never runs an agent. Every agent that acts is
a specialist registered from code, and its allow-list is optional: section 1 lists the grants for
each specialist, to add once the registration cell has created them.

### 2. Give the notebook the key and AWS credentials

Run the setup cell and paste the key at the prompt. It is read with `getpass`, so it is never
echoed and never written into this notebook's saved output — a shared `.ipynb` carries no live
credential. Prefer a file? Put `HIGHFLAME_API_KEY` in a `.env` beside this notebook and the prompt
is skipped; the environment always wins.

| Variable | What it is |
| --- | --- |
| `HIGHFLAME_API_KEY` | **Required.** The team lead's key from step 1. Prompted for if unset. |
| `AWS_PROFILE` | Optional. The AWS profile boto3 uses, for example an SSO profile after `aws sso login --profile <name>`. Unset uses the default credential chain. |
| `BEDROCK_MODEL_ID` | Optional. Unset uses Strands' default Bedrock model; your account needs model access to whichever it is. |
| `AGENT_ROLE_ARN` | Optional. An IAM role each agent assumes for its Bedrock calls, under its own Highflame name, so CloudTrail attributes model calls to the same agent Highflame does. It needs Bedrock invoke permissions and a trust policy your principal may assume. |
| `A2A_PORT` | Optional. The local port the remote agent listens on. Defaults to `9910`. |
| `HIGHFLAME_BASE_URL`, `HIGHFLAME_IDENTITY_URL` | Optional, and set together. A self-hosted deployment. Defaults: `https://api.highflame.ai` and `https://auth.highflame.ai`. |
| `HIGHFLAME_TOKEN_URL` | Optional. Derived as `<identity url>/oauth2/token` unless you set it. |

The remote agent runs inside this kernel on a local port, so nothing needs to be deployed.

Run the install cell once, then restart the kernel.

### 3. Deploy the guardrail policy

Highflame ships its guardrails as policy templates; nothing is enforced until you deploy them. In
Studio, open **Guardrails → Policies** and deploy **Structural PII** (`privacy.defaults`) from the
template catalog, choosing **enforce** in the deploy dialog. It refuses the card number at the
swarm's entry point, and as a pattern detector it runs on every deployment, including one without
the ML detector servers.

Leave the **Default Behavior** strip at the top of that page at *Allow by default*. Tool results
and model replies are permitted by that setting rather than by any template, so with it switched
to Fail Close the first tool result is refused with no policy named.

In [ ]:
%pip install -q -r requirements.txt

In [ ]:
import getpass
import logging
import os
import threading
import time
import uuid
from typing import NamedTuple

import boto3
import httpx
import uvicorn
from a2a.client import ClientConfig
from a2a.client.errors import A2AClientHTTPError
from botocore.exceptions import BotoCoreError, ClientError
from dotenv import load_dotenv
from starlette.middleware.base import BaseHTTPMiddleware
from starlette.responses import JSONResponse

from highflame import APIConnectionError, BlockedError, Highflame
from highflame.integrations.strands import HighflameStrandsHooks
from highflame.zeroid import ToolScope, generate_keypair  # zeroid = Highflame's identity module
from highflame.zeroid.errors import ZeroIDError
from strands import Agent, tool
from strands.agent.a2a_agent import A2AAgent
from strands.models import BedrockModel
from strands.multiagent import Swarm
from strands.multiagent.a2a import A2AServer
from strands.types.exceptions import EventLoopException

load_dotenv()  # .env beside this notebook; real environment variables win

# The team lead's key from Studio, prompted for so it never lands in the saved output.
HIGHFLAME_API_KEY = os.environ.get("HIGHFLAME_API_KEY") or getpass.getpass(
    "Team lead API key from Studio (input hidden): "
).strip()
BEDROCK_MODEL_ID = os.environ.get("BEDROCK_MODEL_ID")
AGENT_ROLE_ARN = os.environ.get("AGENT_ROLE_ARN")
A2A_PORT = int(os.environ.get("A2A_PORT", "9910"))

RUN_ID = uuid.uuid4().hex[:6]  # every identity name carries it, so re-runs never collide
CREATED: list[tuple[str, str]] = []  # (label, id) for the clean-up cell
TOOL_CALLS: list[str] = []  # every tool body appends here, so a cell can prove what ran

ENDPOINTS: dict[str, str] = {}
if os.environ.get("HIGHFLAME_BASE_URL"):
    ENDPOINTS["base_url"] = os.environ["HIGHFLAME_BASE_URL"]
if os.environ.get("HIGHFLAME_IDENTITY_URL"):
    identity_url = os.environ["HIGHFLAME_IDENTITY_URL"].rstrip("/")
    ENDPOINTS["identity_base_url"] = identity_url
    ENDPOINTS["token_url"] = os.environ.get("HIGHFLAME_TOKEN_URL") or f"{identity_url}/oauth2/token"


def highflame_client(**credential: str) -> Highflame:
    """A client on one credential: `api_key=` for a registered agent, `access_token=` for a
    delegated one."""
    return Highflame(**credential, **ENDPOINTS)


base_aws_session = boto3.Session()


def bedrock_model(agent_name: str) -> BedrockModel:
    """The Bedrock model an agent calls, as that agent.

    With AGENT_ROLE_ARN set, the agent assumes the role with its Highflame name as the role session
    name, so CloudTrail records its model calls against the same name Highflame does. Without it,
    every agent calls Bedrock with your credentials and CloudTrail sees one caller.
    """
    session = base_aws_session
    if AGENT_ROLE_ARN:
        credentials = base_aws_session.client("sts").assume_role(
            RoleArn=AGENT_ROLE_ARN, RoleSessionName=agent_name[:64], DurationSeconds=3600
        )["Credentials"]
        session = boto3.Session(
            aws_access_key_id=credentials["AccessKeyId"],
            aws_secret_access_key=credentials["SecretAccessKey"],
            aws_session_token=credentials["SessionToken"],
            region_name=base_aws_session.region_name,
        )
    return BedrockModel(boto_session=session, **({"model_id": BEDROCK_MODEL_ID} if BEDROCK_MODEL_ID else {}))


def bedrock_error(exc: Exception) -> str:
    """The AWS error code, which says what to fix: ExpiredToken, AccessDeniedException, ..."""
    return exc.response["Error"]["Code"] if isinstance(exc, ClientError) else type(exc).__name__


def a2a_text(result) -> str:
    """The remote agent's reply, rejoined.

    A streaming A2A response arrives as many parts, and the converter makes one content
    block per part. `str(result)` then puts a newline after every one, which chops the
    reply into fragments. Joining the blocks with nothing between them restores it.
    """
    return "".join(block.get("text", "") for block in result.message.get("content", []))


highflame_admin = highflame_client(api_key=HIGHFLAME_API_KEY)

# Check the model and the key before anything is registered, so a failure here leaves nothing to
# clean up.
try:
    await Agent(model=bedrock_model("preflight"), callback_handler=None).invoke_async("Reply with the single word OK.")
except (BotoCoreError, ClientError) as exc:
    raise RuntimeError(
        f"Bedrock refused a one-word call ({bedrock_error(exc)}), and nothing has been registered yet. "
        "ExpiredToken or UnrecognizedClient: refresh your AWS credentials. AccessDeniedException: "
        "enable access to the model in the Bedrock console, or set BEDROCK_MODEL_ID."
    ) from exc

NEEDED_SCOPES = {"nhi:manage", "tools:read", "tools:execute", "orders:read", "kb:read", "refunds:read"}
held = set(highflame_admin.tokens.verify(highflame_admin.tokens.issue_api_key(HIGHFLAME_API_KEY).access_token).scopes)
if not NEEDED_SCOPES <= held:
    raise RuntimeError(
        f"The key's credential policy grants {sorted(held) or 'no scopes'}, and this notebook also needs "
        f"{sorted(NEEDED_SCOPES - held)}. List them in the policy (setup step 1), restart the kernel "
        "and re-run. An empty scope list leaves the key nothing to delegate."
    )

print("connected as:", highflame_admin.whoami()["external_id"])

## 1. Register the team

The team lead is the identity you registered in Studio, so nothing is created for it. Four
**specialists** are registered from code, each with a public key so the team lead can delegate to
it, and only the scope its job needs:

| Specialist | Scope | Role |
| --- | --- | --- |
| `triage` | `refunds:read` | Swarm entry point; routes work, and may call the remote refunds agent |
| `orders` | `orders:read` | Swarm member |
| `kb` | `kb:read` | Swarm member |
| `refunds` | `refunds:read` | **Remote** agent served over A2A, running on its own key |

### Authorization, layer two: access grants for each agent

Delegation decides which scopes a specialist's credential carries. What each agent may actually
*do* is its allow-list, set per identity in Studio. The specialists are registered by the cell
below, so they start with the default Access setting and their actions are recorded, not blocked:
the notebook runs without any grants.

To enforce layer two as well, run the cell below, then open each specialist in Studio's Registry
(its name ends with this run's `RUN_ID`), go to its **Policies** page, switch **Access** to
**Enforcing** and add these grants before you run the rest of the notebook:

| Agent | Send prompts | Call tool |
| --- | --- | --- |
| `triage-…` | Allow all | `handoff_to_agent`, `ask_refunds_agent` |
| `orders-…` | Allow all | `handoff_to_agent`, `lookup_order` |
| `kb-…` | Allow all | `handoff_to_agent`, `search_kb` |
| `refunds-…` (remote) | Allow all | `refund_policy` |
| team lead (registered in Studio) | none | none |

- **Send prompts** covers every prompt an agent receives, a hand-off included, and every model
  reply it produces.
- **Call tool** covers the call and its result. Leave the MCP server field empty: these are local
  tools the agents call in process.
- **`handoff_to_agent` is the one people miss.** Strands' `Swarm` gives every member that tool, and
  a hand-off reaches the member's before-tool-call hook like any other tool call. A member without
  the grant cannot hand off at all.
- The team lead needs no grants: it registers and delegates, and never runs an agent.

Leave `ask_refunds_agent` off triage's grants to watch the remote call refused before it leaves the
process: `Refused by Highflame at 'triage': Enterprise Policies Triggered: Authorization Grants — call_tool`.

In [ ]:
# The team lead is the identity you registered in Studio: `highflame_admin` already speaks as it.
team_lead_client = highflame_admin
team_lead_id = team_lead_client.whoami()["external_id"]


class Specialist(NamedTuple):
    external_id: str
    identity_uri: str  # used to delegate to it
    api_key: str  # its own key; only the remote agent, which serves itself, uses it
    private_key_pem: str  # stays here; only the public key went to Highflame
    scopes: str  # the exact scopes to request

    def __repr__(self) -> str:
        # The default NamedTuple repr would print the key and the private key.
        return f"Specialist({self.external_id}, scopes={self.scopes!r}, api_key=<elided>, private_key_pem=<elided>)"


def register_specialist(name: str, domain_scope: str) -> Specialist:
    private_key_pem, public_key_pem = generate_keypair()
    scopes = [ToolScope.READ, ToolScope.EXECUTE, domain_scope]
    reg = highflame_admin.agents.register(
        name=name.title(),
        external_id=f"{name}-{RUN_ID}",
        identity_type="agent",
        sub_type="tool_agent",
        trust_level="first_party",
        framework="strands",
        description="Notebook demo. Safe to delete.",
        allowed_scopes=scopes,
        public_key_pem=public_key_pem,
    )
    CREATED.append((name, reg.agent.id))
    return Specialist(reg.agent.external_id, reg.agent.wimse_uri, reg.api_key, private_key_pem, " ".join(scopes))


triage = register_specialist("triage", "refunds:read")
orders = register_specialist("orders", "orders:read")
kb = register_specialist("kb", "kb:read")
refunds = register_specialist("refunds", "refunds:read")
print("team lead (registered in Studio):", team_lead_id)
print("specialists (registered here)   :", triage.external_id, orders.external_id, kb.external_id, refunds.external_id)

## 2. The remote agent: served over A2A, credential required at the door

The refunds specialist runs as its own service. In production it would be another process or another
company's agent; here it runs on a background thread in this kernel. Three things make it a
Highflame-aware A2A agent:

- **It runs as itself.** Its hooks use its own Highflame key, so every decision it makes is
  attributed to `refunds-…`.
- **It checks who is calling.** A small middleware requires a Highflame credential as a bearer token
  on every A2A message. `tokens.verify()` checks the signature against Highflame's published keys,
  with no network call once the keys are cached, and the middleware then requires the `refunds:read`
  scope. The agent card stays public so callers can discover the agent.
- **It records the caller.** Each accepted request logs the calling identity and the team lead that
  delegated to it, straight from the verified credential. That log is the only link between a caller
  and the remote agent's own decisions, which are recorded under the remote agent's name.

`verify()` does not consult revocation, so a production door should also ask Highflame whether the
credential is still good: Highflame refuses a revoked one with `401 token has been revoked`.

Three refusals, and each one says why:

| Case | Status | What the caller is told |
| --- | --- | --- |
| No bearer token | `401` | `no Highflame credential. send one as: Authorization: Bearer <access token>` |
| A token that does not verify | `401` | `the Highflame credential did not verify. …` |
| A valid token without `refunds:read` | `403` | `kb-… may not call this agent. Missing required scope … The credential holds: …` |

None of the three reaches the model. Each refusal is sent twice: as a JSON body, and as an RFC 6750
`WWW-Authenticate` challenge. The header matters because a streaming A2A client keeps only the
status line and cannot read the body afterwards, so the header is the only half it can still act on.

In [ ]:
REFUND_POLICY = "Refunds are accepted within 30 days of delivery and land in 5 business days."
REQUIRED_SCOPE = "refunds:read"


@tool
def refund_policy(order_id: str) -> str:
    """Return the refund policy that applies to an order."""
    TOOL_CALLS.append("refund_policy")
    return f"Order {order_id}: {REFUND_POLICY}"


refunds_client = highflame_client(api_key=refunds.api_key)  # the remote agent acts as itself
accepted_callers: list[str] = []  # what the door saw, for the telemetry step


def refuse(status: int, code: str, error: str, detail: str) -> JSONResponse:
    """One refusal, said twice.

    The JSON body is for a person, for example a `curl` user. The `WWW-Authenticate`
    header is the RFC 6750 challenge, and it is the half a streaming A2A client can
    still read: that client keeps only the status line and cannot re-read the body.
    """
    challenge = f'Bearer realm="refunds-agent", error="{code}", error_description="{error}. {detail}"'
    if code == "insufficient_scope":
        challenge += f', scope="{REQUIRED_SCOPE}"'
    return JSONResponse({"error": error, "detail": detail}, status_code=status, headers={"WWW-Authenticate": challenge})


class RequireHighflameCredential(BaseHTTPMiddleware):
    """A2A front door: a verified Highflame credential with refunds:read, or no service.

    `401` means "I do not know who you are": no credential, or one that does not verify.
    `403` means "I know who you are, and you may not". Every refusal states the reason,
    and a `403` also names the caller and the scopes its credential holds.
    """

    async def dispatch(self, request, call_next):
        if request.url.path.startswith("/.well-known/"):
            return await call_next(request)  # the agent card is public; that is how callers find us
        bearer = request.headers.get("authorization", "")
        if not bearer.lower().startswith("bearer "):
            return refuse(401, "invalid_request", "no Highflame credential", "send one as: Authorization: Bearer <access token>")
        try:
            caller = refunds_client.tokens.verify(bearer.split(" ", 1)[1])
        except ZeroIDError as exc:
            return refuse(401, "invalid_token", "the Highflame credential did not verify", f"{exc.code}: {exc.message}")
        try:
            caller.require_scope(REQUIRED_SCOPE)
        except ZeroIDError as exc:
            return refuse(
                403,
                "insufficient_scope",
                f"{caller.external_id} may not call this agent",
                f"{exc.message}. The credential holds: {' '.join(caller.scopes) or 'no scopes'}",
            )
        delegated_by = (caller.delegated_by() or "nobody").rsplit("/", 1)[-1]
        accepted_callers.append(f"{caller.external_id} (delegated by {delegated_by})")
        return await call_next(request)


refunds_agent = Agent(
    name="refunds-agent",
    description="Answers refund eligibility and timing questions.",
    model=bedrock_model(refunds.external_id),
    system_prompt="You answer refund questions using refund_policy. Be brief.",
    tools=[refund_policy],
    hooks=[HighflameStrandsHooks(refunds_client, mode="enforce", session_id=f"refunds-agent-{RUN_ID}")],
    trace_attributes={"highflame.agent": refunds.external_id},
    callback_handler=None,
)

a2a_app = A2AServer(agent=refunds_agent, host="127.0.0.1", port=A2A_PORT, enable_a2a_compliant_streaming=True).to_starlette_app()
a2a_app.add_middleware(RequireHighflameCredential)
a2a_server = uvicorn.Server(uvicorn.Config(a2a_app, host="127.0.0.1", port=A2A_PORT, log_level="warning"))
threading.Thread(target=a2a_server.run, daemon=True).start()
while not a2a_server.started:
    time.sleep(0.1)

REFUNDS_AGENT_URL = f"http://127.0.0.1:{A2A_PORT}"
print("A2A agent card:", httpx.get(f"{REFUNDS_AGENT_URL}/.well-known/agent-card.json").json()["name"])

## 3. The swarm: local specialists under delegated credentials

Each swarm member is one of the specialists registered above: the team lead
delegates a short-lived credential to it, and the member's hooks run on that credential. The one
addition is the `triage` member's `ask_refunds_agent` tool, which calls the remote agent over A2A
**presenting triage's own delegated credential** as the bearer token.

A Strands agent keeps its conversation for as long as the object lives, so `run_swarm()` builds a
fresh swarm for every task. Reusing one would carry the previous task into the next, which is how a
refused request ends up quoted in an unrelated answer.

In [ ]:
ORDERS = {"1042": {"status": "delivered", "carrier": "UPS", "delivered_on": "12 days ago", "total": "$129.00"}}


@tool
def lookup_order(order_id: str) -> str:
    """Look up an order by its ID and return status, carrier and delivery date."""
    TOOL_CALLS.append("lookup_order")
    return str(ORDERS.get(order_id, "no such order"))


@tool
def search_kb(query: str) -> str:
    """Search the support knowledge base for policies and how-tos."""
    TOOL_CALLS.append("search_kb")
    return f"KB result for {query!r}: shipping is free over $50; exchanges are handled by the refunds team."


def delegate_to(specialist: Specialist) -> str:
    """A short-lived credential for the specialist, issued by the team lead."""
    return team_lead_client.tokens.delegate_to(
        wimse_uri=specialist.identity_uri, private_key_pem=specialist.private_key_pem, scope=specialist.scopes
    ).access_token


def remote_agent_tool(bearer: str):
    """A tool that asks the remote refunds agent, presenting the caller's credential on the wire."""

    @tool
    async def ask_refunds_agent(question: str) -> str:
        """Ask the refunds agent whether and when an order can be refunded."""
        remote = A2AAgent(
            REFUNDS_AGENT_URL,
            name="refunds-agent",
            client_config=ClientConfig(httpx_client=httpx.AsyncClient(headers={"Authorization": f"Bearer {bearer}"}, timeout=120)),
        )
        # a2a_text, not str(): the model must read one reply, not a column of fragments.
        return a2a_text(await remote.invoke_async(question))

    return ask_refunds_agent


def swarm_member(specialist: Specialist, system_prompt: str, tools: list, credential: str) -> Agent:
    """A guarded member running on a credential delegated to it."""
    return Agent(
        name=specialist.external_id.rsplit("-", 1)[0],  # 'triage', 'orders', 'kb': the names members hand off to
        model=bedrock_model(specialist.external_id),
        system_prompt=system_prompt,
        tools=tools,
        hooks=[HighflameStrandsHooks(highflame_client(access_token=credential), mode="enforce")],
        trace_attributes={"highflame.agent": specialist.external_id},
        callback_handler=None,
    )


# Delegated once here because steps 4 and 5 present the same credential again.
triage_credential = delegate_to(triage)


def build_swarm() -> Swarm:
    triage_agent = swarm_member(
        triage,
        "You are the support triage agent. Hand order-status questions to 'orders' and policy questions to 'kb', "
        "once each. For refund questions use ask_refunds_agent. When a member hands back with its answer, do not "
        "hand off again: give the customer one final answer yourself.",
        [remote_agent_tool(triage_credential)],
        triage_credential,
    )
    return Swarm(
        [
            triage_agent,
            swarm_member(orders, "Look the order up with lookup_order, then hand back to 'triage' with what you found.", [lookup_order], delegate_to(orders)),
            swarm_member(kb, "Search with search_kb, then hand back to 'triage' with what you found.", [search_kb], delegate_to(kb)),
        ],
        entry_point=triage_agent,
        max_handoffs=6,
        max_iterations=8,
        execution_timeout=300.0,
        node_timeout=120.0,
    )


# A Swarm never re-raises what a member raised. It catches the exception, logs the
# traceback, and records the exception on that member's node. So `run_swarm` reads the
# node back instead of relying on `except`, and these two loggers stay quiet: the
# helper below reports the same failure in one line.
logging.getLogger("strands.multiagent.swarm").setLevel(logging.CRITICAL)
logging.getLogger("strands.event_loop.event_loop").setLevel(logging.CRITICAL)


def node_failure(result):
    """The exception a Swarm recorded on a member, with that member's name.

    A refusal inside a member's agent loop -- a tool call, tool result or model reply -- is recorded
    wrapped in Strands' EventLoopException, so it is unwrapped here.
    """
    for node_id, node in (result.results or {}).items():
        if isinstance(node.result, Exception):
            failure = node.result
            return node_id, failure.original_exception if isinstance(failure, EventLoopException) else failure
    return None, None


async def run_swarm(task: str, session_id: str):
    """Run a fresh swarm on one task and print the outcome. Returns None when it was refused or did not finish."""
    try:
        result = await build_swarm().invoke_async(task, invocation_state={"session_id": session_id})
    except Exception as exc:  # a refusal outside a member still raises
        cause = exc.original_exception if isinstance(exc, EventLoopException) else exc
        if isinstance(cause, BlockedError):
            print("Refused by Highflame:", cause.response.policy_reason)
        elif isinstance(cause, APIConnectionError):
            print("Highflame is unreachable. Check your network, or HIGHFLAME_BASE_URL.")
        elif isinstance(cause, (BotoCoreError, ClientError)):
            print(f"The Bedrock call failed ({bedrock_error(cause)}). Check your AWS credentials and model access.")
        else:
            raise
        return None

    # A refusal inside a member arrives here, on its node, not as a raised exception.
    node_id, failure = node_failure(result)
    if isinstance(failure, BlockedError):
        print(f"Refused by Highflame at '{node_id}':", failure.response.policy_reason)
        return None
    if failure is not None:
        print(f"'{node_id}' failed: {type(failure).__name__}: {str(failure)[:200]}")
        return None

    print("status      :", result.status.value)
    print("agents used :", " -> ".join(node.node_id for node in result.node_history))
    if result.status.value != "completed":
        print("The swarm stopped before a final answer (hand-off or time limit). Re-run this cell.")
        return None
    final = result.results.get(result.node_history[-1].node_id) if result.node_history else None
    print("answer      :", str(final.result).strip() if final else None)
    return result

### A question that crosses the team and the wire

Order status comes from the `orders` member via a hand-off. Refund eligibility comes from the remote
agent via `ask_refunds_agent`. Watch the door log: the remote agent saw `triage-…`, delegated by the
team lead.

In [ ]:
await run_swarm("Order 1042 arrived damaged. Where is it now, and can I still get a refund?", f"swarm-{RUN_ID}");
print("\ntool bodies that ran          :", TOOL_CALLS or "none")
print("remote agent accepted callers :", accepted_callers or "none")

### Runtime guardrails apply at the entry point

The swarm's entry member is guarded like any other agent. A prompt that leaks a card number and a
national ID is refused before the model is called, so no hand-off and no remote call ever happens.

This is PII rather than a prompt injection: structural PII is matched by
pattern detectors that run on every deployment, while injection scoring is a model that some
deployments do not run. It uses its own `session_id`, so the incident does not share a conversation
with the ordinary request above.

In [ ]:
calls_before = len(accepted_callers)
if await run_swarm(
    "Here are my details so you can refund order 1042: card 4111-1111-1111-1111, SSN 123-45-6789.",
    f"swarm-pii-{RUN_ID}",
):
    print("\nAllowed: no PII policy is deployed on this account. Deploy Structural PII (setup step 3) and re-run.")
print("remote calls made by this task:", len(accepted_callers) - calls_before)

## 4. Authorization at the A2A boundary

The remote agent enforces identity itself, independent of who is calling. Four direct calls show it,
with no model in the loop for the first three:

1. No credential is refused with `401`.
2. A token that does not verify is refused with `401`.
3. The `kb` specialist's credential verifies but lacks `refunds:read`, so it is refused with `403`.
4. The `triage` credential is accepted.

`door_reason()` reads the reason out of the `WWW-Authenticate` header, because the A2A client keeps
only the status line. Without it, every refusal reads as a bare `HTTP Error 403`, which tells a
developer nothing.

In [ ]:
def door_reason(exc: A2AClientHTTPError) -> str:
    """What the door said, not just the status line.

    The A2A client keeps `status_code` and a generic message, and it discards the body.
    It does chain the underlying httpx error, so the response object survives, but a
    streamed body can no longer be read. The RFC 6750 `WWW-Authenticate` header can.
    """
    response = getattr(getattr(exc, "__cause__", None), "response", None)
    challenge = response.headers.get("www-authenticate", "") if response is not None else ""
    for field in ("error_description", "error"):
        marker = f'{field}="'
        if marker in challenge:
            return challenge.split(marker, 1)[1].split('"', 1)[0]
    return str(exc).splitlines()[0]


async def knock(label: str, bearer: str | None):
    """Call the remote agent with the given credential and report what the door decided."""
    headers = {"Authorization": f"Bearer {bearer}"} if bearer else {}
    remote = A2AAgent(
        REFUNDS_AGENT_URL,
        name="refunds-agent",
        client_config=ClientConfig(httpx_client=httpx.AsyncClient(headers=headers, timeout=120)),
    )
    try:
        result = await remote.invoke_async("Can order 1042 be refunded?")
    except A2AClientHTTPError as exc:
        print(f"{label:18} -> {exc.status_code} refused: {door_reason(exc)}")
        return
    except Exception as exc:
        print(f"{label:18} -> failed: {type(exc).__name__}: {str(exc).splitlines()[0][:90]}")
        return
    print(f"{label:18} -> accepted: {' '.join(a2a_text(result).split())[:90]}...")


await knock("no credential", None)
await knock("bad credential", "not-a-real-token")
await knock("kb credential", delegate_to(kb))
await knock("triage credential", triage_credential)

## 5. Telemetry: who did what, on both sides

Three views of the same run:

- **The credential.** `tokens.verify()` on triage's credential shows who it was issued to, who
  delegated it, and what was granted. The remote agent used exactly this to decide at the door.
- **The door log.** The remote agent's own record of accepted callers — one line per accepted A2A
  message, including the direct call in step 4.
- **Highflame decisions.** A decision made with a member's credential is attributed to that member;
  the remote agent's decisions are attributed to the remote agent. Joining the two is the door log's
  job: Highflame records each side under its own name.

In [ ]:
claims = team_lead_client.tokens.verify(triage_credential)
print("triage credential : issued to", claims.external_id, "| delegated by", (claims.delegated_by() or "?").rsplit("/", 1)[-1])
print("                    scopes", " ".join(claims.scopes), "| depth", claims.delegation_depth)
print("door log          :", ("\n" + " " * 20).join(accepted_callers) or "none")

decision = highflame_client(access_token=triage_credential).guard.evaluate_prompt(
    "Can order 1042 be refunded?", session_id=f"swarm-{RUN_ID}", mode="enforce"
)
print("a triage decision : attributed to", decision.agent_identity.external_id if decision.agent_identity else None, "| request", decision.request_id)

## Clean up

Stop the A2A server and remove the identities this notebook registered from code — the four
specialists. **The team lead you registered in Studio is left alone**, because it is yours: the key in
your `.env` keeps working and the next run reuses it.

`delete()` deactivates rather than erases, so the names stay taken. That is why every name the
notebook creates carries the per-run `RUN_ID`.

In [ ]:
a2a_server.should_exit = True

# Only code-registered identities are in CREATED; the Studio-registered team lead was never added.
for label, identity_id in reversed(CREATED):
    try:
        highflame_admin.agents.delete(identity_id)
        print("deleted:", label)
    except Exception as exc:
        print(f"clean-up skipped for {label}: {str(exc)[:60]}")

## Recap

- **Identity**: the team lead is the agent you registered in Studio; four specialists are registered
  from code. The remote agent runs on its own key; swarm members run on credentials delegated by the
  team lead.
- **Authorization**: each delegated credential carries only its specialist's scope; each
  specialist's allow-list, once Enforcing, decides which tools it may call, hand-offs included; and
  the remote agent re-checks the caller's credential and scope at its own door.
- **Runtime guardrails**: the swarm's entry member refuses a PII leak before anything else happens;
  hand-offs are tool calls and pass the same hooks; content returned from the remote agent is checked
  as a tool result.
- **Telemetry**: the credential names issuer and holder, the door logs verified callers, and every
  Highflame decision names the agent that caused it.

One client, one set of hooks and one error type cover every agent here, local or remote.